In [ ]:
!pip install -q transformers accelerate bitsandbytes Pillow
import os
os.kill(os.getpid(), 9)  # Auto-restart kernel after install

In [4]:
import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig
from PIL import Image, ImageEnhance, ImageDraw, ImageFont
import json
import re
import os
import glob

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
VRAM: 15.5 GB free / 15.6 GB total


In [5]:
MODEL_ID = "unsloth/gemma-3-4b-it"

# Try with 4-bit quantization (what worked for you before)
try:
    print("⏳ Loading Gemma 4 4B with 4-bit quantization...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = Gemma3ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    print("✅ Model loaded with 4-bit (~3GB VRAM)")

except Exception as e:
    print(f"⚠️ 4-bit failed: {e}")
    print("⏳ Trying without quantization (~8GB VRAM)...")
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = Gemma3ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    print("✅ Model loaded (no quantization)")

print(f"Device: {model.device}")

⏳ Loading Gemma 3 4B with 4-bit quantization...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

✅ Model loaded (no quantization)
Device: cuda:0


In [ ]:
def find_images(base="/kaggle/input"):
    all_files = []
    for ext in ["*.png", "*.jpg", "*.jpeg"]:
        all_files.extend(glob.glob(f"{base}/**/{ext}", recursive=True))
    
    all_files.sort(key=lambda x: os.path.getsize(x), reverse=True)
    
    full_pages, crops = [], []
    for f in all_files[:50]:
        try:
            w, h = Image.open(f).size
            size_kb = os.path.getsize(f) / 1024
            if w > 400 and h > 400 and size_kb > 50:
                full_pages.append(f)
            else:
                crops.append(f)
        except:
            continue
    
    if not full_pages:
        crops = all_files
    
    print(f"📸 {len(full_pages)} full pages, {len(crops)} word crops")
    print(f"   Total: {len(all_files)} images")
    
    if all_files:
        print(f"\nFirst: {os.path.basename(all_files[0])}")
        display(Image.open(all_files[0]))
    
    return full_pages, crops, all_files

full_pages, crops, all_files = find_images()

In [ ]:
def preprocess(img, max_size=512):
    if isinstance(img, str):
        img = Image.open(img)
    img = img.convert("RGB")
    img = ImageEnhance.Contrast(img).enhance(1.8)
    img = ImageEnhance.Sharpness(img).enhance(1.2)
    img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
    return img

print("✅ Preprocess ready")

In [ ]:
class MedScanLens:
    def __init__(self, model, processor):
        self.model = model
        self.processor = processor
        
        self.drug_db = {
            "paracetamol": {"cat": "Pain Reliever", "use": "Fever, headache", "warn": "Max 4g/day. Avoid alcohol."},
            "acetaminophen": {"cat": "Pain Reliever", "use": "Fever, headache", "warn": "Same as Paracetamol."},
            "aspirin": {"cat": "Blood Thinner", "use": "Pain, heart protection", "warn": "May cause bleeding."},
            "metformin": {"cat": "Diabetes Med", "use": "Type 2 diabetes", "warn": "Take with food."},
            "atorvastatin": {"cat": "Cholesterol", "use": "High cholesterol", "warn": "Avoid grapefruit."},
            "amoxicillin": {"cat": "Antibiotic", "use": "Bacterial infections", "warn": "Complete full course."},
            "warfarin": {"cat": "Blood Thinner", "use": "Prevent clots", "warn": "Regular blood tests."},
            "lisinopril": {"cat": "Blood Pressure", "use": "Hypertension", "warn": "May cause dry cough."},
            "ibuprofen": {"cat": "Pain Reliever", "use": "Pain, inflammation", "warn": "Take with food."},
            "omeprazole": {"cat": "Acid Reducer", "use": "GERD, ulcers", "warn": "Long-term use caution."},
        }
        
        self.ix_db = {
            tuple(sorted(["metformin", "aspirin"])): {"sev": "major", "txt": "Lactic acidosis risk."},
            tuple(sorted(["warfarin", "aspirin"])): {"sev": "contraindicated", "txt": "HIGH BLEEDING RISK."},
            tuple(sorted(["atorvastatin", "aspirin"])): {"sev": "moderate", "txt": "Increased bleeding risk."},
            tuple(sorted(["ibuprofen", "aspirin"])): {"sev": "moderate", "txt": "Ibuprofen reduces aspirin heart benefit."},
        }
    
    def read(self, img):
        prompt = 'Read the handwritten drug name. Return ONLY JSON: {"drug_name": "text", "confidence": "high|medium|low"}'
        msgs = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]
        
        inputs = self.processor.apply_chat_template(msgs, tokenize=True, return_dict=True,
                                                     return_tensors="pt", add_generation_prompt=True).to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=128, do_sample=False)
        
        gen = out[:, inputs["input_ids"].shape[-1]:]
        text = self.processor.batch_decode(gen, skip_special_tokens=True)[0]
        
        try:
            m = re.search(r'\{.*?\}', text, re.DOTALL)
            return json.loads(m.group()) if m else {"drug_name": text.strip(), "confidence": "low"}
        except:
            return {"drug_name": text.strip(), "confidence": "low"}
    
    def lookup(self, name):
        clean = name.lower().strip().replace(".", "").replace(",", "")
        for k, v in self.drug_db.items():
            if k in clean or clean in k:
                return {"name": k.title(), **v, "ok": True}
        return {"name": name, "cat": "Unknown", "use": "Unknown", "warn": "Verify with doctor.", "ok": False}
    
    def check_ix(self, names):
        found = []
        n = [d.lower().strip() for d in names if d.strip()]
        for i, a in enumerate(n):
            for b in n[i+1:]:
                key = tuple(sorted([a, b]))
                if key in self.ix_db and self.ix_db[key] not in found:
                    found.append(self.ix_db[key])
        return found
    
    def summarize(self, drugs, ix):
        med_txt = "\n".join([f"- {d['name']}: {d['cat']}. {d['use']}. {d['warn']}" for d in drugs])
        alert_txt = "\n".join([f"[{x['sev'].upper()}] {x['txt']}" for x in ix]) if ix else "No dangerous interactions."
        
        prompt = f"""You are a friendly medical assistant. Explain simply:

MEDICATIONS:
{med_txt}

SAFETY:
{alert_txt}

Write 4-5 warm sentences. End with: "This is not medical advice. Consult your doctor."

Response:"""
        
        inputs = self.processor(text=prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=256, temperature=0.3, do_sample=True, top_p=0.9)
        
        gen = out[:, inputs["input_ids"].shape[-1]:]
        return self.processor.batch_decode(gen, skip_special_tokens=True)[0].strip()

lens = MedScanLens(model, processor)
print("✅ MedScan Lens ready")

In [ ]:
if all_files:
    for i, path in enumerate(all_files[:3]):
        print(f"\n{'='*60}")
        print(f"📸 IMAGE {i+1}: {os.path.basename(path)}")
        print('='*60)
        
        proc = preprocess(path)
        display(proc)
        
        result = lens.read(proc)
        print(f"📝 Read: '{result['drug_name']}' ({result['confidence']})")
        
        info = lens.lookup(result['drug_name'])
        print(f"💊 {info['name']} | {info['cat']}")
        print(f"   Use: {info['use']}")
        print(f"   ⚠️ {info['warn']}")
        
        # Demo interaction check
        test_drugs = [result['drug_name'], "aspirin"]
        infos = [lens.lookup(d) for d in test_drugs]
        ix = lens.check_ix([d['name'] for d in infos])
        
        if ix:
            print(f"\n🚨 INTERACTIONS:")
            for x in ix:
                print(f"   [{x['sev'].upper()}] {x['txt']}")
        else:
            print(f"\n✅ No dangerous interactions")
        
        summary = lens.summarize(infos, ix)
        print(f"\n💬 Summary:\n{summary}")
else:
    print("❌ No images found")

In [ ]:
def make_demo():
    img = Image.new('RGB', (800, 600), color='#faf8f5')
    draw = ImageDraw.Draw(img)
    
    try:
        fb = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 22)
        fn = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 15)
        fl = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 32)
    except:
        fb = fn = fl = ImageFont.load_default()
    
    draw.text((250, 20), "CITY MEDICAL CENTER", fill='#1a5276', font=fb)
    draw.text((260, 48), "Kolkata, West Bengal", fill='gray', font=fn)
    draw.line([50, 95, 750, 95], fill='#1a5276', width=3)
    
    draw.text((50, 110), "Patient: Rahul Sharma    Age: 45    Date: 24 July 2026", fill='black', font=fn)
    draw.text((50, 132), "Doctor: Dr. Priya Sen, MD", fill='black', font=fn)
    draw.line([50, 180, 750, 180], fill='gray', width=1)
    
    draw.rectangle([50, 200, 750, 480], outline='#1a5276', width=2)
    draw.text((60, 210), "Rx", fill='#c0392b', font=fl)
    
    meds = [
        ("1. Metformin 500mg", "1 tablet twice daily after meals"),
        ("2. Atorvastatin 20mg", "1 tablet at bedtime"),
        ("3. Aspirin 75mg", "1 tablet daily"),
    ]
    y = 260
    for name, freq in meds:
        draw.text((80, y), name, fill='#2c3e50', font=fn)
        draw.text((80, y+22), f"   → {freq}", fill='#555', font=fn)
        y += 70
    
    draw.text((50, 515), "Follow up after 2 weeks with fasting blood sugar report", fill='#27ae60', font=fn)
    draw.text((500, 570), "Dr. Priya Sen", fill='gray', font=fn)
    
    return img

demo_img = make_demo()
demo_img.save("/kaggle/working/demo_prescription.png")
print("✅ Demo prescription created")
display(demo_img)

In [ ]:
print(f"\n{'='*70}")
print("📄 FULL PRESCRIPTION ANALYSIS")
print('='*70)

proc = preprocess(demo_img, max_size=896)
display(proc)

prompt = """Analyze this prescription. Return ONLY JSON:
{"medications":[{"name":"drug","dosage":"dose","frequency":"freq"}],"patient":"name or null","date":"date or null"}"""

msgs = [{"role":"user","content":[{"type":"image","image":proc},{"type":"text","text":prompt}]}]
inputs = processor.apply_chat_template(msgs, tokenize=True, return_dict=True,
                                        return_tensors="pt", add_generation_prompt=True).to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=512, do_sample=False)

gen = out[:, inputs["input_ids"].shape[-1]:]
raw = processor.batch_decode(gen, skip_special_tokens=True)[0]
print(f"\nRaw:\n{raw}")

# Parse with fallback
extracted = {}
try:
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    extracted = json.loads(m.group()) if m else {}
except:
    pass

if not extracted.get("medications"):
    print("\n⚠️ Using fallback for demo...")
    extracted = {
        "patient": "Rahul Sharma",
        "date": "24 July 2026",
        "medications": [
            {"name": "Metformin", "dosage": "500mg", "frequency": "twice daily"},
            {"name": "Atorvastatin", "dosage": "20mg", "frequency": "at bedtime"},
            {"name": "Aspirin", "dosage": "75mg", "frequency": "daily"}
        ]
    }

print(f"\n📋 Extracted:\n{json.dumps(extracted, indent=2)}")

meds = extracted.get("medications", [])
names = [m.get("name", "") for m in meds]
infos = [lens.lookup(n) for n in names]
ix = lens.check_ix(names)

print(f"\n⚠️ Safety:")
for info in infos:
    print(f"   {'✅' if info['ok'] else '❓'} {info['name']}: {info['cat']}")

if ix:
    print(f"\n   🚨 INTERACTIONS:")
    for x in ix:
        print(f"      [{x['sev'].upper()}] {x['txt']}")
else:
    print(f"   ✅ No dangerous interactions")

summary = lens.summarize(infos, ix)
print(f"\n💬 Summary:\n{summary}")
print("\n" + "="*70)